# 상추 병해 진단

ResNet18으로 정상/병해 3종 분류, YOLOv8로 병반 위치를 탐지합니다.


In [ ]:
# 설정
from pathlib import Path
import json
import random
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 경로
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FARMETRY_DIR = ROOT
BASE_DIR = FARMETRY_DIR / "data" / "samples" / "lettuce_disease"
YOLO_WEIGHTS = FARMETRY_DIR / "yolov8n.pt"
YOLO_ROOT = BASE_DIR / "yolo_dataset"

IMAGE_DIRS = {
    "disease": BASE_DIR / "image" / "image_disease",
    "normal": BASE_DIR / "image" / "image_normal",
}
LABEL_DIRS = {
    "disease": BASE_DIR / "label" / "label_disease",
    "normal": BASE_DIR / "label" / "label_normal",
}

# 라벨
DISEASE_MAP = {0: "정상", 9: "상추균핵병", 10: "상추노균병"}
LABEL_MAP = {0: 0, 9: 1, 10: 2}
CLASS_NAMES = ["정상", "상추균핵병", "상추노균병"]
YOLO_NAMES = CLASS_NAMES.copy()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

print("BASE_DIR:", BASE_DIR, "| exists:", BASE_DIR.exists())
print("YOLO_WEIGHTS:", YOLO_WEIGHTS, "| exists:", YOLO_WEIGHTS.exists())


In [ ]:
# 데이터 로드
def load_label(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def label_to_row(label_path: Path, split: str) -> dict:
    data = load_label(label_path)
    desc, ann = data["description"], data["annotations"]
    bbox = ann["points"][0] if ann.get("points") else {}
    code = ann["disease"]
    return {
        "split": split,
        "image_name": desc["image"],
        "date": desc.get("date"),
        "width": desc["width"], "height": desc["height"],
        "disease": code,
        "disease_name": DISEASE_MAP.get(code, str(code)),
        "xtl": bbox.get("xtl"), "ytl": bbox.get("ytl"),
        "xbr": bbox.get("xbr"), "ybr": bbox.get("ybr"),
    }

rows = []
for split, lbl_dir in LABEL_DIRS.items():
    for p in sorted(lbl_dir.glob("*.json")):
        rows.append(label_to_row(p, split))

df = pd.DataFrame(rows)
df["image_path"] = df.apply(lambda r: IMAGE_DIRS[r["split"]] / r["image_name"], axis=1)

print(f"총 {len(df)}장 |", df["disease_name"].value_counts().to_dict())
df.head(3)


In [ ]:
# EDA
print("클래스 분포:\n", df["disease_name"].value_counts(), sep="")
print("\n날짜:", df["date"].min(), "~", df["date"].max())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (code, name) in zip(axes, [(0,"정상"), (9,"상추균핵병"), (10,"상추노균병")]):
    row = df[df["disease"] == code].iloc[0]
    img = Image.open(row["image_path"]).convert("RGB")
    ax.imshow(img)
    rect = patches.Rectangle(
        (row["xtl"], row["ytl"]), row["xbr"]-row["xtl"], row["ybr"]-row["ytl"],
        linewidth=2, edgecolor="lime", facecolor="none",
    )
    ax.add_patch(rect)
    ax.set_title(name)
    ax.axis("off")
plt.suptitle("클래스별 샘플 + GT bbox")
plt.tight_layout()
plt.show()


In [ ]:
# ResNet18 학습
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

df["label"] = df["disease"].map(LABEL_MAP)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])
print(f"train {len(train_df)} | val {len(val_df)}")

IMG_SIZE, BATCH_SIZE, NUM_EPOCHS, LR = 224, 16, 15, 1e-4
mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2,0.2),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])

class LettuceDS(Dataset):
    def __init__(self, frame, tf=None):
        self.frame = frame.reset_index(drop=True)
        self.tf = tf
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        img = Image.open(r["image_path"]).convert("RGB")
        if self.tf: img = self.tf(img)
        return img, int(r["label"])

train_loader = DataLoader(LettuceDS(train_df, train_tf), BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(LettuceDS(val_df, val_tf), BATCH_SIZE, shuffle=False, num_workers=0)

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 3)
model = model.to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()

history, best_acc, best_state = [], 0.0, None
for ep in range(1, NUM_EPOCHS + 1):
    model.train()
    tl, tc, tn = 0., 0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out = model(x)
        loss = crit(out, y)
        loss.backward(); opt.step()
        tl += loss.item() * len(y)
        tc += (out.argmax(1) == y).sum().item(); tn += len(y)
    model.eval()
    vl, vc, vn = 0., 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x); loss = crit(out, y)
            vl += loss.item() * len(y)
            vc += (out.argmax(1) == y).sum().item(); vn += len(y)
    ta, va = tc/tn, vc/vn
    history.append({"train_acc": ta, "val_acc": va})
    if va > best_acc:
        best_acc, best_state = va, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f"Epoch {ep:02d}/{NUM_EPOCHS} | train {ta:.3f} | val {va:.3f}")

model.load_state_dict(best_state)
print(f"best val acc: {best_acc:.3f}")


In [ ]:
# ResNet18 결과
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

@torch.no_grad()
def predict_all(loader):
    model.eval()
    yt, yp = [], []
    for x, y in loader:
        p = model(x.to(DEVICE)).argmax(1).cpu().numpy()
        yp.extend(p); yt.extend(y.numpy())
    return np.array(yt), np.array(yp)

y_true, y_pred = predict_all(val_loader)
print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([h["train_acc"] for h in history], label="train")
axes[0].plot([h["val_acc"] for h in history], label="val")
axes[0].set_title("학습 곡선"); axes[0].legend(); axes[0].grid(alpha=0.3)
cm = confusion_matrix(y_true, y_pred)
axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks(range(3)); axes[1].set_yticks(range(3))
axes[1].set_xticklabels(CLASS_NAMES, rotation=20, ha="right")
axes[1].set_yticklabels(CLASS_NAMES)
axes[1].set_title("Confusion Matrix")
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, cm[i,j], ha="center", va="center")
plt.tight_layout(); plt.show()

# 정분류 / 오분류 샘플
val_eval = val_df.reset_index(drop=True).copy()
val_eval["pred"] = y_pred
val_eval["pred_name"] = val_eval["pred"].map(dict(enumerate(CLASS_NAMES)))
correct = val_eval[val_eval["label"] == val_eval["pred"]]
wrong = val_eval[val_eval["label"] != val_eval["pred"]]
print(f"정분류 {len(correct)} / 오분류 {len(wrong)}")

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, r) in zip(axes.ravel(), pd.concat([
    correct.groupby("disease_name").sample(1, random_state=SEED),
    wrong.head(3) if len(wrong) else correct.head(3),
]).iterrows()):
    ax.imshow(Image.open(r["image_path"]).convert("RGB"))
    ok = r["disease_name"] == r["pred_name"]
    ax.set_title(f"{'✓' if ok else '✗'} {r['disease_name']}→{r['pred_name']}", fontsize=9,
                 color="green" if ok else "red")
    ax.axis("off")
plt.suptitle("분류 샘플 (위: 정분류 클래스별, 아래: 오분류)")
plt.tight_layout(); plt.show()


In [ ]:
# YOLO 데이터 준비
import shutil, yaml
from sklearn.model_selection import train_test_split

if "df" not in globals():
    raise RuntimeError("데이터 로드 셀을 먼저 실행하세요")

lookup = {(s, f.name.lower()): f for s, d in IMAGE_DIRS.items() for f in d.iterdir() if f.is_file()}

def resolve_path(row):
    k = (row["split"], str(row["image_name"]).lower())
    return lookup.get(k, IMAGE_DIRS[row["split"]] / row["image_name"])

def to_yolo(xtl, ytl, xbr, ybr, w, h):
    vals = [((xtl+xbr)/2)/w, ((ytl+ybr)/2)/h, (xbr-xtl)/w, (ybr-ytl)/h]
    return [max(0, min(1, v)) for v in vals]

yolo_df = df.copy()
yolo_df["image_path"] = yolo_df.apply(resolve_path, axis=1)
yolo_train_df, yolo_val_df = train_test_split(
    yolo_df, test_size=0.2, random_state=SEED, stratify=yolo_df["disease"].map(LABEL_MAP))

def export(frame, split):
    img_d, lbl_d = YOLO_ROOT / "images" / split, YOLO_ROOT / "labels" / split
    img_d.mkdir(parents=True, exist_ok=True); lbl_d.mkdir(parents=True, exist_ok=True)
    for _, r in frame.iterrows():
        src = Path(r["image_path"])
        dst = img_d / src.name
        if not dst.exists(): shutil.copy2(str(src), str(dst))
        cid = LABEL_MAP[int(r["disease"])]
        xc, yc, bw, bh = to_yolo(r["xtl"], r["ytl"], r["xbr"], r["ybr"], r["width"], r["height"])
        (lbl_d / f"{src.stem}.txt").write_text(f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

export(yolo_train_df, "train"); export(yolo_val_df, "val")
yaml_path = YOLO_ROOT / "data.yaml"
yaml_path.write_text(yaml.dump({
    "path": str(YOLO_ROOT.resolve()), "train": "images/train", "val": "images/val",
    "nc": 3, "names": YOLO_NAMES,
}, allow_unicode=True, sort_keys=False), encoding="utf-8")

print(f" 완료 | train {len(yolo_train_df)} | val {len(yolo_val_df)}")
print("yaml:", yaml_path)


In [ ]:
# YOLO 학습
import torch
from ultralytics import YOLO

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
torch.set_num_threads(2)

SKIP_TRAIN = True  # False → 재학습 (CPU ~18분)

if SKIP_TRAIN:
    print("SKIP_TRAIN=True → 학습 건너뜀. 로 이동.")
else:
    if "yaml_path" not in globals():
        yaml_path = YOLO_ROOT / "data.yaml"
    print("학습 시작 (epoch=5, CPU)...")
    res = YOLO(str(YOLO_WEIGHTS)).train(
        data=str(yaml_path), epochs=5, imgsz=416, batch=4,
        device="cpu", workers=0,
        project=str(FARMETRY_DIR / "runs" / "detect"),
        name="lettuce_disease_yolo", patience=5, seed=SEED,
        amp=False, cache=False, plots=False, verbose=True,
    )
    print("완료:", res.save_dir)


In [ ]:
# YOLO 결과 확인
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FARMETRY_DIR = ROOT

pts = sorted(FARMETRY_DIR.glob("runs/detect/lettuce_disease_yolo*/weights/best.pt"))
if not pts:
    raise FileNotFoundError("best.pt 없음 → 에서 SKIP_TRAIN=False로 학습")
best_pt = pts[-1]
print("weight:", best_pt)

csv = best_pt.parent.parent / "results.csv"
if not csv.exists():
    raise FileNotFoundError(f"results.csv 없음: {csv}")

last = pd.read_csv(csv).iloc[-1]
print(f"\n=== YOLO 결과 (epoch {int(last['epoch'])}) ===")
print(f"  mAP50    : {last['metrics/mAP50(B)']:.3f}")
print(f"  mAP50-95 : {last['metrics/mAP50-95(B)']:.3f}")
print(f"  Precision: {last['metrics/precision(B)']:.3f}")
print(f"  Recall   : {last['metrics/recall(B)']:.3f}")
print("\n 완료 → 병징 위치 그림은  실행")


In [ ]:
# YOLO 병반 시각화
import gc
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch
from ultralytics import YOLO

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
torch.set_num_threads(1)
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FARMETRY_DIR = ROOT
YOLO_ROOT = FARMETRY_DIR / "data" / "samples" / "lettuce_disease" / "yolo_dataset"
YOLO_NAMES = ["정상", "상추균핵병", "상추노균병"]
THUMB = 384
YOLO_IMGSZ = 320

# ===== 다른 샘플 보기 (아래만 바꾸고 셀 재실행) =====
RANDOM_SEED = 42       # 숫자 바꾸면 클래스별 랜덤 샘플 변경 (예: 0, 7, 99)
N_PER_CLASS = 1        # 클래스당 몇 장 볼지 (1~3 권장, 많을수록 느림)
CLASS_INDEX = None     # 한 클래스만: 0=정상, 1=균핵병, 2=노균병 / 전체는 None
TARGET_FILE = None     # 특정 파일 1장만: "파일명.jpg" / 사용 안 하면 None
# TARGET_FILE = "V006_77_1_09_05_03_12_1_0177e_20201014_74.jpg"

pts = sorted(FARMETRY_DIR.glob("runs/detect/lettuce_disease_yolo*/weights/best.pt"))
if not pts:
    raise FileNotFoundError("best.pt 없음")
best_pt = pts[-1]
print("model:", best_pt.name)

val_img = YOLO_ROOT / "images" / "val"
val_lbl = YOLO_ROOT / "labels" / "val"
if not val_img.exists():
    raise FileNotFoundError("yolo_dataset 없음 — YOLO 데이터 준비 셀을 먼저 실행")

def resize_save(src: Path) -> tuple[Path, Image.Image]:
    im = Image.open(src).convert("RGB")
    im.thumbnail((THUMB, THUMB))
    tmp = Path(tempfile.gettempdir()) / f"yolo_{src.stem}.jpg"
    im.save(tmp, quality=85)
    return tmp, im

def draw_boxes(ax, pil_img, boxes, names):
    ax.imshow(pil_img)
    if boxes is None:
        return
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]
        cid = int(box.cls.item())
        conf = float(box.conf.item())
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, lw=2, ec="red", fc="none"))
        ax.text(x1, max(y1 - 4, 0), f"{names[cid]} {conf:.2f}", color="red", fontsize=8,
                bbox=dict(facecolor="white", alpha=0.7, pad=1))

import random
random.seed(RANDOM_SEED)

def list_by_class(cid):
    out = []
    for img in sorted(val_img.glob("*.*")):
        lbl = val_lbl / f"{img.stem}.txt"
        if lbl.exists() and lbl.read_text(encoding="utf-8").strip().startswith(str(cid)):
            out.append((img, YOLO_NAMES[cid], lbl))
    return out

samples = []
if TARGET_FILE:
    p = val_img / TARGET_FILE
    lbl = val_lbl / f"{p.stem}.txt"
    if not p.exists():
        raise FileNotFoundError(f"없음: {p}")
    cid = int(lbl.read_text(encoding="utf-8").strip().split()[0])
    samples = [(p, YOLO_NAMES[cid], lbl)]
else:
    classes = [CLASS_INDEX] if CLASS_INDEX is not None else range(3)
    for cid in classes:
        pool = list_by_class(cid)
        if not pool:
            continue
        random.shuffle(pool)
        samples.extend(pool[:N_PER_CLASS])

print(f"시각화 {len(samples)}장 | seed={RANDOM_SEED}")
for img_p, cname, _ in samples:
    print(" -", cname, "|", img_p.name)
yolo = YOLO(str(best_pt))

for img_p, cname, lbl_p in samples:
    _, xc, yc, bw, bh = map(float, lbl_p.read_text(encoding="utf-8").strip().split())
    tmp, disp = resize_save(img_p)
    w, h = disp.size
    gx1, gy1 = (xc - bw/2) * w, (yc - bh/2) * h
    gw, gh = bw * w, bh * h

    pred = yolo.predict(str(tmp), conf=0.25, device="cpu", imgsz=YOLO_IMGSZ, verbose=False)[0]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(disp)
    axes[0].add_patch(patches.Rectangle((gx1, gy1), gw, gh, lw=2, ec="lime", fc="none"))
    axes[0].set_title(f"GT: {cname}"); axes[0].axis("off")
    draw_boxes(axes[1], disp.copy(), pred.boxes, YOLO_NAMES)
    axes[1].set_title("YOLO 예측"); axes[1].axis("off")
    plt.tight_layout(); plt.show()
    plt.close(fig)
    gc.collect()

del yolo; gc.collect()
print("\n 완료")
